# Loss Functions from Scratch: Cross-Entropy, BCE, Focal and Dice — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

`F.cross_entropy` is in almost every training script, yet few people can say exactly what it computes once `weight`, `ignore_index` and `label_smoothing` are combined. Interviewers like to ask about exactly that, together with the numerical-stability questions behind every loss that takes logits:

- **Cross-entropy** with `ignore_index` (the void label 255 in semantic segmentation, the padding label -100 in captioning and LLM training), **class weights** (long-tailed data), and **label smoothing** (Inception-v3, the DeiT/ViT recipes). It takes **K-dim inputs** `(N, C, H, W)` for per-pixel classification.
- **BCE with logits** for multi-label classification, GAN discriminators and the SigLIP image–text loss. `pos_weight` trades recall against precision.
- **Sigmoid focal loss** (Lin et al., *Focal Loss for Dense Object Detection*, RetinaNet). A dense detector scores ~100k anchors per image and almost all of them are easy background. Focal loss stops these easy negatives from dominating the gradient.
- **Soft Dice loss**, the standard for binary and medical segmentation, where the foreground may be 1% of the pixels.

Implement all four from scratch with **exactly** PyTorch's / torchvision's semantics. The grader checks values **and gradients** against `F.cross_entropy` and `F.binary_cross_entropy_with_logits`, including logits as large as $10^4$.

---

### Requirements

1. **`cross_entropy(logits, target, weight=None, ignore_index=-100, label_smoothing=0.0) -> Tensor`** (0-dim) must equal `F.cross_entropy(logits, target, weight=weight, ignore_index=ignore_index, label_smoothing=label_smoothing)` with the default `reduction="mean"`.
   - `logits` is `(N, C)` or `(N, C, d1, ..., dk)`, with classes on **dim 1**. `target` is `(N,)` or `(N, d1, ..., dk)`, holding int64 class indices or `ignore_index`.
   - With $M$ = the non-ignored positions, $w_c$ = the class weights (all 1 if `weight is None`), $\log p_{i,c} = x_{i,c} - \operatorname{logsumexp}_{c'} x_{i,c'}$ and $\varepsilon$ = `label_smoothing`:
   $$\mathcal{L} = (1-\varepsilon)\,\frac{\sum_{i\in M} w_{y_i}\,(-\log p_{i,y_i})}{\sum_{i\in M} w_{y_i}} \;+\; \frac{\varepsilon}{C}\,\frac{\sum_{i\in M}\sum_{c} w_c\,(-\log p_{i,c})}{\sum_{i\in M} w_{y_i}}$$
     This is PyTorch's exact formula. The weighted mean divides by the **sum of the target weights**, not by $N$, and the smoothing term is weighted per class and shares the same denominator. Equivalently, it is the cross-entropy against the smoothed target $q = (1-\varepsilon)\,\text{onehot}(y) + \varepsilon/C$, weighted by $w_c$ and normalised by $\sum_{i \in M} w_{y_i}$.
   - Only class-index targets are required (no probability targets). If every target is ignored the result is $0/0$ = NaN, as in PyTorch (not graded).
2. **`binary_cross_entropy_with_logits(logits, targets, pos_weight=None, reduction="mean")`** must equal `F.binary_cross_entropy_with_logits`:
   $$\ell = -\big[p_w\, y\log\sigma(x) + (1-y)\log(1-\sigma(x))\big] = (1-y)\,x + \big(1 + (p_w-1)\,y\big)\,\operatorname{softplus}(-x)$$
   - `targets` may be soft (values in $[0, 1]$). `pos_weight` broadcasts against the last dimension (shape `(C,)` for multi-label). `reduction` is `"mean"`, `"sum"` or `"none"`.
   - Values and gradients must be exact for $|x|$ up to $10^4$, **and the gradient must be right at exactly $x = 0$**: a zero-initialised classifier head outputs exactly 0 on its first step.
3. **`sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="mean")`** must equal `torchvision.ops.sigmoid_focal_loss`. `targets` are 0/1 floats with the same shape as `logits`.
   $$p = \sigma(x),\quad p_t = p\,y + (1-p)(1-y),\quad \alpha_t = \alpha\,y + (1-\alpha)(1-y),\quad \mathrm{FL} = \alpha_t\,(1-p_t)^{\gamma}\,\underbrace{(-\log p_t)}_{\text{BCE with logits}}$$
   - $\alpha$ weights the **positives** and $1-\alpha$ the negatives. `alpha < 0` disables the $\alpha_t$ factor entirely.
   - `reduction="mean"` averages over **all** elements. (RetinaNet divides the summed loss by the number of positive anchors, but that happens outside this function.)
   - With `gamma=0` and `alpha<0` it reduces to BCE-with-logits. It must stay finite for $|x|$ up to $10^4$.
4. **`dice_loss(logits, targets, eps=1.0)`** is the soft Dice loss for binary masks `(N, 1, H, W)`. Any `(N, ...)` shape works: everything except the batch dimension is summed.
   $$p = \sigma(x),\qquad \mathcal{L} = \frac{1}{N}\sum_{n=1}^{N}\left(1 - \frac{2\sum_j p_{nj}\,t_{nj} + \epsilon}{\sum_j p_{nj} + \sum_j t_{nj} + \epsilon}\right)$$
   - Dice is computed **per sample** and then averaged. `eps` appears in the numerator **and** the denominator, so an empty mask predicted empty gives a loss of 0.

---

### Constraints

- ✅ `torch.logsumexp`, `torch.logaddexp`, `F.softplus`, `F.logsigmoid`, `torch.sigmoid`, `gather`, `F.one_hot`.
- ❌ `F.cross_entropy`, `F.nll_loss`, `F.binary_cross_entropy(_with_logits)`, `nn.*Loss`, `torchvision.ops`. The grader traps the `F.*` losses. Also avoid `F.log_softmax` / `torch.log_softmax`: building log-softmax from `logsumexp` is the point.
- ✅ Vectorised: no Python loops over samples, pixels or classes.

---

<details>
  <summary>💡 Hint</summary>

  - `softmax` underflows to exactly 0 for logits about 104 below the max (float32), and `log(0) = -inf`. Can you write $\log p_c$ without ever computing $p_c$?
  - For K-dim inputs, first get the logits into an `(M, C)` matrix and the targets into an `(M,)` vector. Which axis has to move before you flatten?
  - `gather` cannot index with -100. Replace ignored targets with any valid class, then cancel their contribution with a mask.
  - $-\log\sigma(x) = \log(1 + e^{-x})$ and $-\log(1-\sigma(x)) = x + \log(1 + e^{-x})$. Which built-in computes $\log(e^a + e^b)$ stably?
  - For focal loss, reuse your BCE (with `reduction="none"`) as the $-\log p_t$ term.

</details>

---

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)

# Why "from logits" matters: the naive route overflows/underflows long before 1e4.
big = torch.tensor([[1e4, 0.0, -1e4]])
print("log(softmax(x))   :", torch.log(torch.softmax(big, dim=1)))
print("x - logsumexp(x)  :", big - torch.logsumexp(big, dim=1, keepdim=True))
print("log(sigmoid(-1e4)):", torch.log(torch.sigmoid(torch.tensor(-1e4))))

# Shared test data
N, C = 16, 5
logits = torch.randn(N, C) * 3
target = torch.randint(0, C, (N,))
target_ign = target.clone()
target_ign[::4] = -100                                # every 4th target is ignored
class_w = torch.tensor([0.2, 1.0, 3.0, 0.7, 1.5])
seg_logits = torch.randn(2, C, 6, 7) * 2              # (N, C, H, W) per-pixel classification
seg_target = torch.randint(0, C, (2, 6, 7))
seg_target[0, :2] = 255                               # "void" pixels


def assert_same(got, want, what, atol=1e-5):
    assert torch.isfinite(got).all(), f"{what}: NaN/Inf"
    assert torch.allclose(got, want, atol=atol, rtol=1e-5), \
        f"{what}: got {got.detach()}, expected {want.detach()}"

In [ ]:
def cross_entropy(logits, target, weight=None, ignore_index=-100, label_smoothing=0.0):
    """F.cross_entropy with class-index targets and mean reduction.

    Args:
        logits: (N, C) or (N, C, d1, ..., dk) raw scores.
        target: (N,) or (N, d1, ..., dk) int64 class indices in [0, C), or ignore_index.
        weight: optional (C,) per-class weights.
        ignore_index (int): target value that contributes neither to the loss nor to the mean.
        label_smoothing (float): eps in [0, 1].

    Returns:
        0-dim tensor.
    """
    C = logits.shape[1]
    x = logits.movedim(1, -1).reshape(-1, C)             # (M, C): class axis LAST, then flatten
    t = target.reshape(-1)                               # (M,), same (n, d1, ..., dk) order
    # stable log-softmax: subtracting logsumexp never exponentiates a large positive number
    logp = x - torch.logsumexp(x, dim=1, keepdim=True)   # (M, C)

    valid = t != ignore_index                            # (M,) bool
    t_safe = torch.where(valid, t, torch.zeros_like(t))  # gather() needs a valid index everywhere
    w = torch.ones(C, dtype=logp.dtype, device=logp.device) if weight is None else weight.to(logp)
    w_t = w[t_safe] * valid                              # weight of each target; 0 if ignored
    denom = w_t.sum()                                    # sum of weights of non-ignored targets

    nll = -logp.gather(1, t_safe[:, None]).squeeze(1)    # (M,) -log p_{i, y_i}
    loss = (w_t * nll).sum() / denom
    if label_smoothing > 0:
        # PyTorch: (1-eps) * NLL + (eps/C) * sum_c w_c * (-log p_c), with the same denominator
        smooth = -(logp * w).sum(dim=1) * valid          # (M,)
        loss = (1 - label_smoothing) * loss + (label_smoothing / C) * smooth.sum() / denom
    return loss

In [ ]:
assert_same(cross_entropy(logits, target), F.cross_entropy(logits, target), "plain")
assert_same(cross_entropy(logits, target_ign), F.cross_entropy(logits, target_ign), "ignore_index")
assert_same(cross_entropy(logits, target, weight=class_w),
            F.cross_entropy(logits, target, weight=class_w), "class weights")
assert_same(cross_entropy(logits, target_ign, weight=class_w, label_smoothing=0.2),
            F.cross_entropy(logits, target_ign, weight=class_w, label_smoothing=0.2),
            "weights + ignore + smoothing")
assert_same(cross_entropy(seg_logits, seg_target, ignore_index=255, label_smoothing=0.1),
            F.cross_entropy(seg_logits, seg_target, ignore_index=255, label_smoothing=0.1),
            "(N, C, H, W) segmentation logits")
assert_same(cross_entropy(big, torch.tensor([2])), F.cross_entropy(big, torch.tensor([2])), "1e4 logits")

x = logits.clone().requires_grad_()
cross_entropy(x, target_ign, weight=class_w, label_smoothing=0.1).backward()
x_ref = logits.clone().requires_grad_()
F.cross_entropy(x_ref, target_ign, weight=class_w, label_smoothing=0.1).backward()
assert_same(x.grad, x_ref.grad, "gradient")
print("cross_entropy matches F.cross_entropy")

In [ ]:
def binary_cross_entropy_with_logits(logits, targets, pos_weight=None, reduction="mean"):
    """F.binary_cross_entropy_with_logits, numerically stable.

    -[pw*y*log(sigmoid(x)) + (1-y)*log(1-sigmoid(x))] = (1-y)*x + (1 + (pw-1)*y) * softplus(-x),
    using -log(sigmoid(x)) = softplus(-x) and -log(1-sigmoid(x)) = x + softplus(-x).
    """
    # softplus(-x) = log(e^0 + e^-x). logaddexp evaluates it as max(0,-x) + log1p(exp(-|x|))
    # (never exponentiates a large number) AND differentiates it correctly at x = 0, where
    # the hand-written max/abs version gets the subgradient -1 instead of -0.5.
    softplus_neg = torch.logaddexp(torch.zeros_like(logits), -logits)
    if pos_weight is None:
        loss = (1 - targets) * logits + softplus_neg
    else:
        loss = (1 - targets) * logits + (1 + (pos_weight - 1) * targets) * softplus_neg
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss

In [ ]:
bx = torch.tensor([1e4, -1e4, 1e4, -1e4, 30.0, -30.0, 2.0, -1.0, 0.0])
by = torch.tensor([1.0, 1.0, 0.0, 0.0, 0.3, 0.7, 1.0, 0.0, 1.0])
pw = torch.tensor(3.0)
for kw in [dict(), dict(pos_weight=pw), dict(reduction="none"), dict(pos_weight=pw, reduction="sum")]:
    xa, xb = bx.clone().requires_grad_(), bx.clone().requires_grad_()
    got = binary_cross_entropy_with_logits(xa, by, **kw)
    want = F.binary_cross_entropy_with_logits(xb, by, **kw)
    assert_same(got, want, f"BCE {kw}")
    got.sum().backward()
    want.sum().backward()
    assert_same(xa.grad, xb.grad, f"BCE gradient {kw} (note x = 0 is in the batch)")

ml_logits, ml_targets = torch.randn(8, 4) * 4, (torch.rand(8, 4) > 0.7).float()   # multi-label
ml_pw = torch.tensor([1.0, 2.0, 5.0, 0.5])
assert_same(binary_cross_entropy_with_logits(ml_logits, ml_targets, pos_weight=ml_pw),
            F.binary_cross_entropy_with_logits(ml_logits, ml_targets, pos_weight=ml_pw), "multi-label")
print("binary_cross_entropy_with_logits matches F.binary_cross_entropy_with_logits")

In [ ]:
def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="mean"):
    """Sigmoid focal loss (RetinaNet), torchvision.ops.sigmoid_focal_loss semantics."""
    p = torch.sigmoid(logits)
    ce = binary_cross_entropy_with_logits(logits, targets, reduction="none")  # -log(p_t), stable
    p_t = p * targets + (1 - p) * (1 - targets)          # probability given to the true label
    loss = ce * (1 - p_t) ** gamma                        # easy examples (p_t -> 1) fade out
    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss

In [ ]:
fx, fy = torch.randn(6, 50) * 3, (torch.rand(6, 50) < 0.1).float()
# gamma = 0 and alpha < 0: plain BCE
assert_same(sigmoid_focal_loss(fx, fy, alpha=-1, gamma=0.0), F.binary_cross_entropy_with_logits(fx, fy),
            "gamma=0, alpha<0 must equal BCE")
# alpha weights positives, 1 - alpha negatives: a positive and a negative with the same p_t
pair_x, pair_y = torch.tensor([2.0, -2.0]), torch.tensor([1.0, 0.0])
l = sigmoid_focal_loss(pair_x, pair_y, alpha=0.25, gamma=2.0, reduction="none")
assert torch.allclose(l[0] / l[1], torch.tensor(0.25 / 0.75)), "alpha_t must be alpha for y=1, 1-alpha for y=0"
# the modulating factor down-weights easy examples far more than hard ones
easy, hard = torch.tensor([4.0]), torch.tensor([-1.0])          # both positives
ratio_easy = sigmoid_focal_loss(easy, torch.ones(1), alpha=-1) / F.binary_cross_entropy_with_logits(easy, torch.ones(1))
ratio_hard = sigmoid_focal_loss(hard, torch.ones(1), alpha=-1) / F.binary_cross_entropy_with_logits(hard, torch.ones(1))
assert ratio_easy < 1e-3 < 0.5 < ratio_hard, (ratio_easy, ratio_hard)
# finite at extreme logits, and "mean" averages over all elements
ex = torch.tensor([1e4, -1e4, 1e4, -1e4]).requires_grad_()
ey = torch.tensor([1.0, 1.0, 0.0, 0.0])
out = sigmoid_focal_loss(ex, ey, reduction="none")
out.sum().backward()
assert torch.isfinite(out).all() and torch.isfinite(ex.grad).all(), "NaN/Inf at |logits| = 1e4"
assert_same(sigmoid_focal_loss(fx, fy), sigmoid_focal_loss(fx, fy, reduction="none").mean(), "mean")
try:                                                    # optional cross-check
    from torchvision.ops import sigmoid_focal_loss as tv_focal
    assert_same(sigmoid_focal_loss(fx, fy), tv_focal(fx, fy, reduction="mean"), "vs torchvision")
    print("(matches torchvision.ops.sigmoid_focal_loss)")
except ImportError:
    pass
print("sigmoid_focal_loss tests passed")

In [ ]:
def dice_loss(logits, targets, eps=1.0):
    """Soft Dice loss: mean over samples of 1 - (2*sum(p*t) + eps) / (sum(p) + sum(t) + eps)."""
    p = torch.sigmoid(logits).flatten(1)                 # (N, C*H*W)
    t = targets.flatten(1).to(p.dtype)
    inter = (p * t).sum(dim=1)                           # soft |P intersect T| per sample
    dice = (2 * inter + eps) / (p.sum(dim=1) + t.sum(dim=1) + eps)
    return (1 - dice).mean()

In [ ]:
# hand-computed: p = [0.5, 0.5], t = [1, 0]  ->  dice = (2*0.5 + 1) / (1 + 1 + 1) = 2/3
assert_same(dice_loss(torch.zeros(1, 1, 1, 2), torch.tensor([[[[1.0, 0.0]]]])), torch.tensor(1 / 3), "hand example")
masks = (torch.rand(3, 1, 8, 8) < 0.3).float()
masks[2] = 0                                             # an empty mask
perfect = (2 * masks - 1) * 20                           # +20 on the mask, -20 elsewhere
assert dice_loss(perfect, masks).item() < 1e-4, "perfect prediction -> ~0 (empty mask predicted empty too)"
assert dice_loss(-perfect, masks).item() > 0.6, "inverted prediction -> close to 1"
# per-sample, not global: one sample with a huge mask must not drown out one with a tiny mask
a = torch.zeros(2, 1, 4, 4)
a[0] = 1                                                 # sample 0: everything foreground
a[1, 0, 0, 0] = 1                                        # sample 1: a single pixel
pred = torch.full((2, 1, 4, 4), -20.0)
pred[0] = 20.0                                           # sample 0 right, sample 1 completely missed
assert abs(dice_loss(pred, a).item() - 0.5 * (1 - 1 / 2)) < 1e-3, "average the per-sample Dice losses"
xd = (torch.randn(2, 1, 8, 8) * 1e4).requires_grad_()
dice_loss(xd, masks[:2]).backward()
assert torch.isfinite(xd.grad).all(), "gradients must stay finite at extreme logits"
print("dice_loss tests passed")

In [ ]:
# Why focal and Dice exist: two class-imbalance stories.
torch.manual_seed(0)
# 1) Dense detection: 10,000 anchors, 1% positives. The classifier already rejects most
#    background confidently (easy negatives), but there are SO many of them.
n_pos, n_neg = 100, 9900
x = torch.cat([torch.randn(n_pos) * 1.5,                 # positives: still uncertain (hard)
               torch.randn(n_neg) * 1.0 - 4.0])          # negatives: mostly easy, p ~ 0.02
y = torch.cat([torch.ones(n_pos), torch.zeros(n_neg)])
for name, per_elem in [("BCE", binary_cross_entropy_with_logits(x, y, reduction="none")),
                       ("focal (a=.25, g=2)", sigmoid_focal_loss(x, y, reduction="none"))]:
    share = per_elem[n_pos:].sum() / per_elem.sum()
    print(f"{name:>20}: easy-negative share of the total loss = {share:.1%}")

# 2) Segmentation with a 2% foreground: predicting "all background" looks fine to BCE.
mask = torch.zeros(4, 1, 32, 32)
mask[:, :, 10:14, 10:15] = 1                             # 20 of 1024 pixels
all_background = torch.full_like(mask, -4.0)
print(f"all-background prediction: BCE = {binary_cross_entropy_with_logits(all_background, mask):.3f}"
      f"   Dice loss = {dice_loss(all_background, mask):.3f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("losses-from-scratch", cross_entropy=cross_entropy,
      binary_cross_entropy_with_logits=binary_cross_entropy_with_logits,
      sigmoid_focal_loss=sigmoid_focal_loss, dice_loss=dice_loss)

### Why it works

- **Log-softmax via log-sum-exp.** $\log p_c = x_c - \log\sum_j e^{x_j} = (x_c - m) - \log\sum_j e^{x_j - m}$ with $m = \max_j x_j$. The largest exponent is $e^0 = 1$, so nothing overflows, and $\log p_c$ is computed *directly* instead of as the log of an underflowed 0. Its gradient is $\text{softmax}(x) - \text{onehot}(y)$, which is bounded in $[-1, 1]$. With `log(softmax(x))`, any class more than ~104 below the max (float32) has $p_c = 0$ exactly, so $\log p_c = -\infty$ and the backward pass computes $0\cdot\infty = \text{NaN}$.
- **Weighted mean.** Dividing by $\sum_{i\in M} w_{y_i}$ makes the loss a true weighted *average*. Rescaling all weights leaves it unchanged, and ignored positions drop out of both the numerator and the denominator. Smoothing then mixes in $\varepsilon/C$ of the uniform distribution over *all* $C$ classes (the true class gets $1-\varepsilon+\varepsilon/C$). PyTorch weights that term per class with the same denominator, so the result is the weighted cross-entropy against the smoothed target $q$.
- **BCE with logits.** $-\log\sigma(x) = \log(1+e^{-x}) = \operatorname{softplus}(-x)$ and $-\log(1-\sigma(x)) = x + \operatorname{softplus}(-x)$. Substituting gives $(1-y)x + (1+(p_w-1)y)\operatorname{softplus}(-x)$. The gradient is $\sigma(x)(1 + (p_w-1)y) - p_w y$, which reduces to $\sigma(x) - y$ without `pos_weight` and never blows up. **The subgradient trap:** the textbook stable form $\max(-x,0) + \log(1+e^{-|x|})$ is smooth as a function, but autograd differentiates the pieces. At $x = 0$ it takes slope $-1$ from `clamp` and slope $0$ from `abs` (sign(0) = 0), which gives $-1$ instead of $-\sigma(0) = -0.5$. `torch.logaddexp(0, -x)` and `F.softplus` compute the same value with a correct derivative. PyTorch's own kernel sidesteps the issue with a hand-written backward ($\sigma(x) - y$).
- **Focal loss.** The modulating factor $(1-p_t)^\gamma$ is ≈ 1 for misclassified examples and → 0 for easy ones. With $\gamma = 2$, an example at $p_t = 0.9$ is down-weighted 100× and one at $p_t = 0.99$ 10,000×. The thousands of easy negatives then stop dominating the sum (see the demo above: their share of the loss collapses). Note that $\alpha = 0.25$ weights the *positives* **less**. Once $\gamma$ has removed the easy negatives, the paper found it best to also tone down the positives. RetinaNet also initialises the classification bias to $-\log((1-\pi)/\pi)$ with $\pi = 0.01$, so the first iterations are not swamped by background loss.
- **Dice** is a differentiable relaxation of the F1 / Dice overlap $2|P\cap T|/(|P|+|T|)$ with soft $p$. It is a *ratio* within each image, so it does not care how many background pixels there are. That is why it handles a 2% foreground well where per-pixel BCE is happy to predict "all background" (demo above). Computing it per sample keeps large-object images from drowning out small-object ones. $\epsilon$ in the numerator and denominator defines Dice = 1 for an empty mask predicted empty, and keeps the gradient finite.

### Complexity

All four losses are $O(N \cdot C)$ (or $O(N\cdot HW)$) time and memory, a few elementwise kernels plus a reduction. Cross-entropy materialises an `(M, C)` log-prob matrix. For an LLM with a 128k vocabulary that matrix, in fp32, can be the largest activation in the network. Fused / chunked cross-entropy kernels compute the loss and its gradient block by block without ever storing the full matrix.

### Common bugs

- `torch.log(torch.softmax(x))` or `torch.log(torch.sigmoid(x))`: NaN/Inf at large logits. Clamping the probabilities "fixes" the NaN, but it caps the loss at $-\log(\epsilon)$ and zeroes the gradient exactly where the model is most wrong.
- Calling `gather` with `-100` targets (an index error), or masking the ignored terms to 0 but still dividing by $N$.
- A weighted loss divided by $N$ instead of $\sum w_{y_i}$. A smoothing term that is unweighted, or that uses $\varepsilon/(C-1)$ on the wrong classes only (a different, also common, definition that does not match PyTorch).
- `logits.reshape(-1, C)` on `(N, C, H, W)`: it runs and returns garbage, because classes get mixed with pixels. Move the class axis last first (`movedim(1, -1)`).
- Hand-written stable BCE with a wrong gradient at exactly 0, or `pos_weight` applied to the whole loss of positive samples (only identical for hard 0/1 targets).
- Focal: $\alpha$ applied to the negatives, $(1-p)^\gamma$ instead of $(1-p_t)^\gamma$, ignoring the `alpha < 0` switch, or normalising by the number of positives inside the function.
- Dice: forgetting the sigmoid, a global (batch-level) Dice, $\epsilon$ only in the denominator (empty masks can then never reach loss 0), or mixing in the squared-denominator variant ($\sum p^2 + \sum t^2$, from V-Net) without saying so.

### Interview follow-ups

1. **Why does `nn.CrossEntropyLoss` take logits and not probabilities?** Fusing log-softmax and NLL lets the implementation use the log-sum-exp trick and the simple gradient $\text{softmax} - \text{onehot}$. Taking probabilities would force a `log` of values that may have underflowed to 0. A model that ends in `Softmax` and feeds `CrossEntropyLoss` applies softmax twice. It still trains, but badly: the loss cannot go below $\log(1 + (C-1)/e)$.
2. **What does label smoothing do to calibration and to knowledge distillation?** It stops the logits from growing without bound (the optimum logit gap becomes finite), which usually improves calibration and acts as a regulariser. Müller et al. (2019, *When does label smoothing help?*) showed that it pulls same-class penultimate features into tight, equidistant clusters. That erases the inter-class similarity information ("dark knowledge") that distillation relies on, so teachers trained with label smoothing make worse teachers even when they are more accurate.
3. **When would you choose focal loss over class-weighted BCE?** Class weights rescale *all* examples of a class, including the easy ones, and still let a mass of easy negatives dominate. Focal loss reweights each example by its *difficulty*, which suits dense detection (one-stage detectors, the classification branches of FCOS/RetinaNet) and other problems with extreme imbalance plus many trivial examples. On balanced classification it mostly costs a little calibration, with no gain.
4. **Dice vs cross-entropy for segmentation: why do many pipelines use both?** Per-pixel CE gives smooth, well-behaved gradients but is dominated by the background when the foreground is tiny. Dice optimises the overlap metric directly and ignores the background count, but its gradients are noisier (a ratio) and unstable for very small objects. `CE + Dice` (nnU-Net's default) combines CE's stability with Dice's imbalance robustness.
5. **How would you implement cross-entropy for a 128k-token vocabulary without running out of memory?** Never materialise the `(tokens, vocab)` fp32 log-probs. Chunk over tokens (or over the vocabulary with an online log-sum-exp, like flash attention's softmax), compute the loss and $\text{softmax} - \text{onehot}$ per chunk, and write the gradient straight into the logits' grad buffer. The `ignore_index` mask also lets you skip padding and prompt tokens entirely.
6. **Why does BCE's `pos_weight` change precision/recall and not just the loss scale?** Weighting the positive term by $p_w$ shifts the loss minimiser for a given input from $p = P(y{=}1\mid x)$ to the odds-scaled $p_w P/(p_w P + 1 - P)$. The model systematically over-predicts positives (higher recall, lower precision), as if the positives had been oversampled $p_w$ times.